In [2]:
import pytbl
import panel
import pandas as pd
import warnings
import numpy as np
from datetime import datetime
from tqdm import tqdm
from joblib import Parallel, delayed
from concurrent.futures import ThreadPoolExecutor
from IPython import get_ipython
from IPython.display import display, Javascript
from IPython import get_ipython
from IPython import get_ipython
ip = get_ipython()
path = None
if '__vsc_ipynb_file__' in ip.user_ns:
    path = ip.user_ns['__vsc_ipynb_file__']
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
warnings.filterwarnings("ignore")
begDate    = "20200130"
endData    = "20201123"
sampleDate = "20210421"

from busdates import PortableBusDates
from tqdm import tqdm
pbd = PortableBusDates()
tradingdays = pbd.get_range(begDate, endData)
nodatelist = pbd.get_range("20201123", "20210421")
preface_tradingdays = pbd.get_range(pbd.prev_date(begDate, 122), begDate) + tradingdays
alpha_name = path.split('.')[0].split('/')[-1]

In [19]:
import os
import pandas as pd

def compute_fr_correlation(fr_df: pd.DataFrame, folder_path='./fr_save/'):
    target_columns = [
        'FR_all_ret0e_shift',
        'FR_all_ret1h_shift',
        'FR_10:30:00_ret0e_shift',
        'FR_10:30:00_ret1h_shift'
    ]

    records = []

    for filename in os.listdir(folder_path):
        if filename.endswith('.parquet'):
            file_path = os.path.join(folder_path, filename)
            df = pd.read_parquet(file_path)

            # 确保只计算两边都存在的列
            common_cols = [col for col in target_columns if col in df.columns and col in fr_df.columns]

            # 对齐索引
            aligned_fr_df, aligned_df = fr_df[common_cols].align(df[common_cols], join='inner')

            # 计算每列的相关系数
            row = {'filename': filename}
            for col in common_cols:
                row[col] = aligned_fr_df[col].corr(aligned_df[col])
            
            # 添加平均相关性
            row['mean_corr'] = sum(row[col] for col in common_cols) / len(common_cols)
            records.append(row)

    result_df = pd.DataFrame(records)

    # # 输出 top5 和 bottom5
    # print("Top 5 correlation files:")
    # print(result_df.sort_values('mean_corr', ascending=False).head(5))

    # print("\nBottom 5 correlation files:")
    # print(result_df.sort_values('mean_corr', ascending=True).head(5))

    return result_df


In [32]:
folder_path= './fr_save_old'
for filename in os.listdir(folder_path):
    if filename.endswith('.parquet'):
        file_path = os.path.join(folder_path, filename)
        sample_fr_df = pd.read_parquet(f'./fr_save_old/{filename}')
        result_df = compute_fr_correlation(sample_fr_df, './fr_save_old')
        print(filename, '-----\n', result_df.iloc[:,0:4:2].sort_values(by=['FR_all_ret1h_shift']))

dense_AvgavgPrc_minusclose.parquet -----
                                         filename  FR_all_ret1h_shift
9                           dense_Qty_BS.parquet           -0.730290
2            upshallowtran_Num_ratioFull.parquet           -0.169638
12           upshallowtran_Qty_ratioFull.parquet           -0.156128
7                      ImpactPIN_BSratio.parquet           -0.126292
10                       dense_Qtydratio.parquet           -0.118849
8                  Around3s_add_BSratioQ.parquet           -0.087298
1                 invImpactPIN_BSmallQty.parquet           -0.058480
6                 Impactret_sumQtydratio.parquet           -0.012939
5                    OutAggres_Qtydratio.parquet           -0.005096
4   maxhist20Interval_Qty_BaddSdFull_log.parquet            0.001405
13                   Impactret_sumQty_BS.parquet            0.010717
3           ret_maxhist20Interval_AvgPrc.parquet            0.012341
11                inverstransP_BSmallQty.parquet            0

In [31]:
folder_path= './fr_save_old'
for filename in os.listdir(folder_path):
    if filename.endswith('.parquet'):
        file_path = os.path.join(folder_path, filename)
        sample_fr_df = pd.read_parquet(f'./fr_save_old/{filename}')
        result_df = compute_fr_correlation(sample_fr_df, './fr_save_old')
        print(filename, '-----\n', result_df.iloc[:,:2].sort_values(by=['FR_all_ret0e_shift']))

dense_AvgavgPrc_minusclose.parquet -----
                                         filename  FR_all_ret0e_shift
9                           dense_Qty_BS.parquet           -0.751024
8                  Around3s_add_BSratioQ.parquet           -0.238841
4   maxhist20Interval_Qty_BaddSdFull_log.parquet           -0.190191
11                inverstransP_BSmallQty.parquet           -0.183308
2            upshallowtran_Num_ratioFull.parquet           -0.113208
12           upshallowtran_Qty_ratioFull.parquet           -0.099629
10                       dense_Qtydratio.parquet           -0.015389
13                   Impactret_sumQty_BS.parquet            0.041394
5                    OutAggres_Qtydratio.parquet            0.116682
3           ret_maxhist20Interval_AvgPrc.parquet            0.146616
1                 invImpactPIN_BSmallQty.parquet            0.213537
6                 Impactret_sumQtydratio.parquet            0.220072
7                      ImpactPIN_BSratio.parquet            0